# Milestone 12 Multimodal Architecture Design & Sanity Validation

Milestone 12 establishes the core CLIP‑style multimodal architecture that aligns
molecule representations and phenotype image representations in a shared latent space.
This milestone focuses exclusively on **model structure**, **projection heads**, and
**sanity‑check validation** using synthetic data. No real molecules or real images are
used here — that begins in Milestone 13.

---

## Objectives

- Define modular encoders for molecules and images.
- Implement projection heads that map both modalities into a common embedding space.
- Build a multimodal wrapper model with a CLIP‑style contrastive loss.
- Validate the architecture using random tensors:
  - forward pass works
  - embeddings have correct shapes
  - similarity matrix is computed correctly
  - contrastive loss returns a scalar
  - gradients flow and the model is trainable

---

## Architecture Components

### **1. Molecule Encoder**
A lightweight molecular encoder that converts RDKit `Mol` objects or feature vectors
into fixed‑dimensional embeddings (e.g., 256‑dim).  
This encoder is intentionally simple for architectural validation.

### **2. Image Encoder**
A ResNet18 backbone that produces fixed‑dimensional image embeddings.  
The final fully‑connected layer is replaced with a projection to the desired embedding size.

### **3. Multimodal Alignment Head**
Two projection layers (one for molecules, one for images) map both embeddings into a
shared latent space.  
Outputs are L2‑normalized to enable cosine‑similarity‑based contrastive learning.

### **4. Contrastive Loss**
A symmetric CLIP‑style contrastive loss encourages matching molecule–image pairs to be
close together while pushing apart non‑matching pairs.

---

## Sanity Checks Performed

To validate the architecture, we:

- Generated random molecule and image embeddings.
- Passed them through the multimodal model.
- Computed the contrastive loss.
- Verified that:
  - the loss is non‑zero for batch sizes ≥ 2,
  - gradients propagate correctly,
  - optimizer steps update model parameters.

A typical output from the sanity check:



In [15]:
# ------------------------------------------------------------
# Core PyTorch imports
# ------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models   # vision backbone (ResNet18, etc.)
from rdkit import Chem           # molecule handling (SMILES → Mol)

smiles = "CCO"   # ethanol example
mol = Chem.MolFromSmiles(smiles)


In [35]:
class SimpleMolEncoder(nn.Module):
    def __init__(self, max_atomic_num=100, emb_dim=256):
        super().__init__()
        # A single linear layer that maps a 100‑dim atom‑count vector
        # into a learned molecular embedding of size emb_dim.
        self.fc = nn.Linear(max_atomic_num, emb_dim)

    def forward(self, mol):
        # Create a zero vector to store counts of each atomic number.
        # Index 0–99 correspond to atomic numbers 0–99.
        atom_counts = torch.zeros(100)

        # Loop over all atoms in the RDKit molecule.
        for atom in mol.GetAtoms():
            Z = atom.GetAtomicNum()      # atomic number (e.g., C=6, O=8)
            if Z < 100:                  # safety check: ignore exotic atoms
                atom_counts[Z] += 1.0    # increment count for this element

        # Add a batch dimension so the tensor becomes shape (1, 100)
        atom_counts = atom_counts.unsqueeze(0)

        # Pass the atom‑count vector through the linear layer
        # and apply ReLU to introduce non‑linearity.
        emb = F.relu(self.fc(atom_counts))

        # Return the final molecular embedding (shape: [1, emb_dim])
        return emb


In [36]:
class ImageEncoder(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        # Load a pretrained ResNet18 backbone.
        # This gives us a strong feature extractor for phenotype images.
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # The original ResNet18 classification head outputs 1000 classes.
        # We retrieve the number of input features to that final layer.
        in_features = self.backbone.fc.in_features

        # Replace the final fully-connected layer with a new one
        # that outputs an embedding of size emb_dim (e.g., 256).
        # This converts ResNet18 into a general-purpose feature encoder.
        self.backbone.fc = nn.Linear(in_features, emb_dim)

    def forward(self, img):
        # If the input image is a single tensor of shape (C, H, W),
        # add a batch dimension so it becomes (1, C, H, W).
        if img.dim() == 3:
            img = img.unsqueeze(0)

        # Pass the image through the ResNet backbone and apply ReLU
        # to introduce non-linearity in the final embedding.
        emb = F.relu(self.backbone(img))

        # Return the final image embedding (shape: [batch_size, emb_dim]).
        return emb


In [37]:
class MultimodalModel(nn.Module):
    def __init__(self, emb_dim=256, proj_dim=128):
        super().__init__()
        # Linear projection that maps molecule embeddings (emb_dim)
        # into a shared latent space of dimension proj_dim.
        self.mol_proj = nn.Linear(emb_dim, proj_dim)

        # Linear projection that maps image embeddings (emb_dim)
        # into the same shared latent space (proj_dim).
        self.img_proj = nn.Linear(emb_dim, proj_dim)

    def forward(self, mol_emb, img_emb):
        # Project molecule embeddings and L2-normalize them.
        # Normalization ensures cosine similarity behaves properly.
        z_mol = F.normalize(self.mol_proj(mol_emb), dim=1)

        # Project image embeddings and L2-normalize them.
        z_img = F.normalize(self.img_proj(img_emb), dim=1)

        # Return aligned embeddings for both modalities.
        return z_mol, z_img

    def contrastive_loss(self, z_mol, z_img, temperature=0.1):
        # Compute pairwise similarity matrix between all molecule
        # and image embeddings using dot product scaled by temperature.
        sim_matrix = torch.matmul(z_mol, z_img.T) / temperature

        # Number of samples in the batch.
        batch_size = sim_matrix.size(0)

        # Labels represent correct molecule–image pairs:
        # sample i should match sample i.
        labels = torch.arange(batch_size).to(sim_matrix.device)

        # Cross-entropy loss treating each row of sim_matrix
        # as logits predicting the correct image for each molecule.
        loss_mol_to_img = nn.CrossEntropyLoss()(sim_matrix, labels)

        # Symmetric loss: each column predicts the correct molecule.
        loss_img_to_mol = nn.CrossEntropyLoss()(sim_matrix.T, labels)

        # Final CLIP-style contrastive loss is the average of both directions.
        return (loss_mol_to_img + loss_img_to_mol) / 2


In [ ]:
model = MultimodalModel()


In [38]:
# Generate a batch of 4 random molecule embeddings,
# each of dimensionality 256. This simulates the output
# of a molecule encoder during architecture testing.
mol_emb = torch.randn(4, 256)

# Generate a batch of 4 random image embeddings,
# also 256‑dimensional. This simulates the output
# of an image encoder before real data integration.
img_emb = torch.randn(4, 256)

# Pass both embeddings through the multimodal model.
# The model projects each modality into a shared latent
# space (proj_dim = 128 by default) and L2‑normalizes them.
z_mol, z_img = model(mol_emb, img_emb)


In [39]:
# Compute the CLIP‑style contrastive loss between the projected
# molecule embeddings (z_mol) and image embeddings (z_img).
# This loss encourages matching pairs to have high similarity
# and non‑matching pairs to have low similarity.
loss = model.contrastive_loss(z_mol, z_img)

# Display the scalar loss value so we can verify that the
# architecture is behaving correctly (non‑zero for batch ≥ 2).
loss


tensor(1.0599, grad_fn=<DivBackward0>)

In [40]:
# Create an Adam optimizer that will update all parameters
# inside the multimodal model. The learning rate (1e-3)
# controls how large each update step is during training.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Reset any previously accumulated gradients.
# This is essential because PyTorch accumulates gradients
# by default across backward passes.
optimizer.zero_grad()

# Backpropagate the contrastive loss through the model.
# This computes gradients for all parameters involved
# in producing z_mol, z_img, and the final loss.
loss.backward()

# Apply one optimization step: update model weights
# using the gradients computed above.
optimizer.step()


In [41]:
# Run a small training loop for 3 steps to verify that:
# - the model produces valid embeddings,
# - the contrastive loss decreases or behaves sensibly,
# - gradients flow correctly,
# - optimizer updates the model parameters.

for i in range(3):
    # Generate a batch of 4 random molecule embeddings (256‑dim each).
    # These simulate the output of a molecule encoder during architecture testing.
    mol_emb = torch.randn(4, 256)

    # Generate a batch of 4 random image embeddings (256‑dim each).
    # These simulate the output of an image encoder before real data integration.
    img_emb = torch.randn(4, 256)

    # Forward pass: project both modalities into the shared latent space.
    z_mol, z_img = model(mol_emb, img_emb)

    # Compute the CLIP‑style contrastive loss between molecule and image embeddings.
    loss = model.contrastive_loss(z_mol, z_img)

    # Reset gradients from the previous iteration.
    optimizer.zero_grad()

    # Backpropagate the loss to compute gradients for all model parameters.
    loss.backward()

    # Update model parameters using the Adam optimizer.
    optimizer.step()

    # Print the loss for this training step to monitor behavior.
    print(f"Step {i+1} — Loss: {loss.item():.4f}")


Step 1 — Loss: 1.2744
Step 2 — Loss: 1.1924
Step 3 — Loss: 0.8528


In [42]:
# Instantiate the molecule encoder.
# This creates a SimpleMolEncoder object that converts RDKit Mol objects
# into fixed‑dimensional molecular embeddings (default: 256‑dim).
mol_encoder = SimpleMolEncoder()

# Instantiate the image encoder.
# This creates an ImageEncoder object built on a pretrained ResNet18 backbone,
# projecting phenotype images into 256‑dim embeddings.
img_encoder = ImageEncoder()


In [43]:
from PIL import Image
import torchvision.transforms as T

# Preprocessing pipeline for ResNet18.
# Every pretrained ResNet expects:
# - 224×224 resolution
# - pixel values normalized with ImageNet mean/std
# - channel order: RGB
transform = T.Compose([
    T.Resize((224, 224)),          # Resize image to the expected input size
    T.ToTensor(),                  # Convert PIL image → PyTorch tensor (C×H×W)
    T.Normalize(mean=[0.485, 0.456, 0.406],   # Standard ImageNet normalization
                std=[0.229, 0.224, 0.225])
])

# Use an existing image from your project folder.
# This ensures the encoder is tested on a real phenotype image
# instead of synthetic random tensors.
img_path = "C:/Users/anjal/OneDrive/Documents/phenotypic-virtual-screening/dual_library_VS.png"

# Load the image using PIL and ensure it is in RGB mode.
img = Image.open(img_path).convert("RGB")

# Apply preprocessing transforms and encode the image.
# After transform: shape is (3, 224, 224)
# After encoder: shape is (1, 256)
img = transform(img)
img_emb = img_encoder(img)


In [44]:
# Encode the molecule using the SimpleMolEncoder.
# This converts the RDKit Mol object into a 256‑dimensional molecular embedding.
mol_emb = mol_encoder(mol)

# Encode the image using the ImageEncoder.
# The image tensor (after preprocessing) is passed through ResNet18
# and projected into a 256‑dimensional image embedding.
img_emb = img_encoder(img)

# Forward pass through the multimodal model.
# Both embeddings are projected into a shared latent space (128‑dim by default)
# and L2‑normalized to enable cosine‑similarity‑based contrastive learning.
z_mol, z_img = model(mol_emb, img_emb)


In [45]:
# Generate random embeddings for both modalities.
# These simulate the output of the molecule and image encoders
# during architecture validation (before using real data).
mol_emb = torch.randn(4, 256)
img_emb = torch.randn(4, 256)

# Forward pass through the multimodal model.
# The model projects both embeddings into a shared latent space
# and L2-normalizes them for cosine-similarity-based alignment.
z_mol, z_img = model(mol_emb, img_emb)

# Compute the CLIP-style contrastive loss between the two modalities.
# This verifies that the loss function behaves correctly and returns
# a meaningful scalar value for batch sizes ≥ 2.
loss = model.contrastive_loss(z_mol, z_img)

# Print the loss to confirm the architecture is functioning as expected.
print(loss)


tensor(2.0130, grad_fn=<DivBackward0>)
